In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [7]:

# Note: Using a TDD-inspired approach. Testing the forward and backward pass 
# on a single neuron to verify gradient flow.x1 = Value(2.0, label='x1')
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

b = Value(6.8813735870195432, label='b')


x1w1 = x1*w1
x1w1.label = 'x1*w1'

x2w2 = x2*w2
x2w2.label = 'x2*w2'

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = 'x1w1 + x2w2'

n = x1w1x2w2 + b
n.label = 'n'

o = n.tanh() # I think I will stick to initial tanh activation function, makes sense to me
o.label = 'o'

# Trigger the micrograd engine
o.backward()

print(f"x1 gradient: {x1.grad}")
print(f"x2 gradient: {x2.grad}")
print(f"w1 gradient: {w1.grad}")
print(f"w2 gradient: {w2.grad}")

x1 gradient: -1.4999999999999996
x2 gradient: 0.4999999999999999
w1 gradient: 0.9999999999999998
w2 gradient: 0.0


In [12]:
class Value:
    def __init__(self, data=0.0, label='optional', children=()):
        self.data = data
        self.label = label
        self._backward = lambda: None
        self.grad = 0
        self._children = set(children)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        
        result = Value(self.data * other.data, label = '*', children=(self, other))

        def _backward():
            self.grad += other.data * result.grad
            other.grad += self.data * result.grad

        result._backward = _backward
        return result

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        
        result = Value(self.data + other.data, label = '+', children=(self, other))

        def _backward():
            self.grad += result.grad
            other.grad += result.grad

        result._backward = _backward
        return result

    def tanh(self):
        result = Value(math.tanh(self.data), label = 'tanh', children=(self, ))

        def _backward():
            self.grad += (1 - result.data**2) * result.grad

        result._backward = _backward
        return result

    def backward(self):
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children: 
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        
        self.grad = 1.0 # I think this should be 1, Carpathy used grad as 1. Make sense for multiplication DC/DC = 1
        for node in reversed(topo):
            #here I finally trigger the _backward function I was passing in every single __method__
            node._backward()

    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


In [4]:
import torch
import random

In [8]:
x1 = torch.Tensor([2.00]).double()
x1.requires_grad = True
x2 = torch.Tensor([0.00]).double()
x2.requires_grad = True
w1 = torch.Tensor([-3.00]).double()
w1.requires_grad = True
w2 = torch.Tensor([1.00]).double()
w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()
b.requires_grad = True

n = x1*w1 + x2*w2 + b

o = torch.tanh(n)

o.backward()

print("o: ", o.data.item())
print("x1 grad: ", x1.grad.item())
print("x2 grad: ", x2.grad.item())
print("w1 grad: ", w1.grad.item())
print("w2 grad: ", w2.grad.item())

o:  0.7071066904050358
x1 grad:  -1.5000003851533106
x2 grad:  0.5000001283844369
w1 grad:  1.0000002567688737
w2 grad:  0.0


In [13]:
class Neuron:    
    def __init__(self, number_of_inputs):
        self.w = [Value(random.uniform(-1, 1)) for inpt in range (number_of_inputs)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        
        result = sum(xi * wi for xi, wi in zip(x, self.w)) + self.b

        return result.tanh()
        

x = [2.0, 3.0]
n = Neuron(2)
print("Labas")
print(n(x))

Labas
Value(data=-0.9943348351722908, grad=0)
